# Configuration

In [ ]:
import json
import requests
from datetime import datetime

import pandas as pd

from pyspark.sql import functions as F
from pyspark.sql.window import Window

bucket = "aws-logs-664024262614-us-east-1"

base_url = "https://e6uw49pbah.execute-api.us-east-1.amazonaws.com/dev"
token = "STUDENT_TOKEN_2026"
station_id = "GDN_01"
limit = 1000

raw_path = f"s3://aws-logs-664024262614-us-east-1/project12/raw/weather_raw.json"
clean_path = f"s3://aws-logs-664024262614-us-east-1/project12/processed/weather_clean/"
features_path = f"s3://aws-logs-664024262614-us-east-1/project12/processed/weather_features/"
scored_path = f"s3://aws-logs-664024262614-us-east-1/project12/output/rain_risk_scored/"
metrics_path = f"s3://aws-logs-664024262614-us-east-1/project12/output/evaluation_metrics/"

# Loading raw data
Fetching raw weather data batches from the REST API using the authorization token.

In [ ]:
url = f"{base_url}/weather/batch"
headers = {
    "Authorization": f"Bearer {token}"
}
params = {
    "station_id": station_id,
    "limit": limit
}

response = requests.get(url, headers=headers, params=params, timeout=30)

if response.status_code != 200:
    raise RuntimeError(f"API error: {response.status_code}, {response.text}")

data = response.json()

if isinstance(data, dict) and "body" in data:
    body = data["body"]
    if isinstance(body, str):
        data = json.loads(body)
    else:
        data = body

if isinstance(data, dict):
    for key in ["items", "data", "measurements", "records", "results"]:
        if key in data:
            data = data[key]
            break

if isinstance(data, dict):
    data = [data]

print("Downloaded records:", len(data))
print(data[0])

In [ ]:
import boto3
import json

s3 = boto3.client("s3")

raw_key = "project12/raw/weather_raw.json"

s3.put_object(Bucket=bucket,
              Key=raw_key,
              Body=json.dumps(data, indent=4),
              ContentType="application/json")

raw_path = f"s3://{bucket}/{raw_key}"

print("Raw data saved to:", raw_path)

In [ ]:
raw_df_pandas = pd.DataFrame(data)

print(raw_df_pandas.head())
print(raw_df_pandas.dtypes)
print("Rows:", len(raw_df_pandas))

In [ ]:
df = spark.createDataFrame(raw_df_pandas)

print("Spark DataFrame created")
print(df.columns)

# Cleaning data
Selecting required columns, casting data types, and handling physical anomalies (e.g., clipping negative wind speeds) using PySpark.

In [ ]:
from pyspark.sql import functions as F

required_columns = [
    "timestamp",
    "station_id",
    "temperature",
    "humidity",
    "pressure",
    "wind_speed",
    "wind_direction",
    "rain_mm",
    "cloud_cover"
]

missing_columns = [col for col in required_columns if col not in df.columns]

if missing_columns:
    raise ValueError(f"Missing columns: {missing_columns}. Available columns: {df.columns}")

clean_df = df.select(required_columns)

clean_df = clean_df.withColumn("timestamp", F.to_timestamp("timestamp"))

numeric_columns = [
    "temperature",
    "humidity",
    "pressure",
    "wind_speed",
    "wind_direction",
    "rain_mm",
    "cloud_cover"
]

for col_name in numeric_columns:
    clean_df = clean_df.withColumn(col_name, F.col(col_name).cast("double"))

clean_df = clean_df.dropna(subset=["timestamp", "station_id"])
clean_df = clean_df.dropDuplicates(["timestamp", "station_id"])

clean_df = clean_df.withColumn(
    "humidity",
    F.when(F.col("humidity") < 0, 0)
     .when(F.col("humidity") > 100, 100)
     .otherwise(F.col("humidity"))
)

clean_df = clean_df.withColumn(
    "cloud_cover",
    F.when(F.col("cloud_cover") < 0, 0)
     .when(F.col("cloud_cover") > 100, 100)
     .otherwise(F.col("cloud_cover"))
)

clean_df = clean_df.withColumn(
    "rain_mm",
    F.when(F.col("rain_mm") < 0, 0).otherwise(F.col("rain_mm"))
)

clean_df = clean_df.withColumn(
    "wind_speed",
    F.when(F.col("wind_speed") < 0, 0).otherwise(F.col("wind_speed"))
)

clean_df = clean_df.orderBy("station_id", "timestamp")

print("Cleaning transformations prepared.")
print("Columns:", clean_df.columns)

In [ ]:
clean_pandas = raw_df_pandas.copy()

required_columns = [
    "timestamp",
    "station_id",
    "temperature",
    "humidity",
    "pressure",
    "wind_speed",
    "wind_direction",
    "rain_mm",
    "cloud_cover"
]

clean_pandas = clean_pandas[required_columns]

clean_pandas["timestamp"] = pd.to_datetime(clean_pandas["timestamp"], errors="coerce")

numeric_columns = [
    "temperature",
    "humidity",
    "pressure",
    "wind_speed",
    "wind_direction",
    "rain_mm",
    "cloud_cover"
]

for col in numeric_columns:
    clean_pandas[col] = pd.to_numeric(clean_pandas[col], errors="coerce")

clean_pandas = clean_pandas.dropna(subset=["timestamp", "station_id"])
clean_pandas = clean_pandas.drop_duplicates(subset=["timestamp", "station_id"])
clean_pandas = clean_pandas.sort_values(["station_id", "timestamp"]).reset_index(drop=True)

clean_pandas["humidity"] = clean_pandas["humidity"].clip(lower=0, upper=100)
clean_pandas["cloud_cover"] = clean_pandas["cloud_cover"].clip(lower=0, upper=100)
clean_pandas["rain_mm"] = clean_pandas["rain_mm"].clip(lower=0)
clean_pandas["wind_speed"] = clean_pandas["wind_speed"].clip(lower=0)

clean_key = "project12/processed/weather_clean.csv"

s3.put_object(
    Bucket=bucket,
    Key=clean_key,
    Body=clean_pandas.to_csv(index=False),
    ContentType="text/csv"
)

clean_path = f"s3://{bucket}/{clean_key}"

print("Clean data saved to:", clean_path)
print("Rows:", len(clean_pandas))
print(clean_pandas.head())

# Feature Engineering
Creating moving averages (rolling windows), calculating pressure differences, and defining the target variable (`will_rain_next`).

In [ ]:
import numpy as np

features_pandas = clean_pandas.copy()

features_pandas["timestamp"] = pd.to_datetime(features_pandas["timestamp"])
features_pandas = features_pandas.sort_values(["station_id", "timestamp"]).reset_index(drop=True)

grouped = features_pandas.groupby("station_id", group_keys=False)

features_pandas["humidity_roll_mean_3"] = grouped["humidity"].rolling(3, min_periods=1).mean().reset_index(level=0, drop=True)
features_pandas["cloud_cover_roll_mean_3"] = grouped["cloud_cover"].rolling(3, min_periods=1).mean().reset_index(level=0, drop=True)
features_pandas["wind_speed_roll_mean_3"] = grouped["wind_speed"].rolling(3, min_periods=1).mean().reset_index(level=0, drop=True)
features_pandas["rain_last_3_sum"] = grouped["rain_mm"].rolling(3, min_periods=1).sum().reset_index(level=0, drop=True)

features_pandas["pressure_previous"] = grouped["pressure"].shift(1)
features_pandas["pressure_change"] = features_pandas["pressure"] - features_pandas["pressure_previous"]

features_pandas["rain_previous"] = grouped["rain_mm"].shift(1)
features_pandas["rain_next"] = grouped["rain_mm"].shift(-1)

features_pandas["will_rain_next"] = (features_pandas["rain_next"] > 0).astype(int)

features_pandas = features_pandas.dropna(
    subset=["pressure_previous", "rain_previous", "rain_next"]
).reset_index(drop=True)

features_key = "project12/processed/weather_features.csv"

s3.put_object(
    Bucket=bucket,
    Key=features_key,
    Body=features_pandas.to_csv(index=False),
    ContentType="text/csv"
)

features_path = f"s3://{bucket}/{features_key}"

print("Feature data saved to:", features_path)
print("Rows:", len(features_pandas))
print(features_pandas.head())

In [ ]:
features_key = "project12/processed/weather_features.csv"

s3.put_object(
    Bucket=bucket,
    Key=features_key,
    Body=features_pandas.to_csv(index=False),
    ContentType="text/csv"
)

features_path = f"s3://{bucket}/{features_key}"

print("Feature data saved to:", features_path)
print("Rows:", len(features_pandas))

In [ ]:
def add_normalized_column(df, source_col, target_col):
    stats = df.agg(
        F.min(source_col).alias("min_value"),
        F.max(source_col).alias("max_value")
    ).collect()[0]

    min_value = stats["min_value"]
    max_value = stats["max_value"]

    if min_value == max_value:
        return df.withColumn(target_col, F.lit(50.0))

    return df.withColumn(
        target_col,
        100 * (F.col(source_col) - F.lit(min_value)) / (F.lit(max_value) - F.lit(min_value))
    )

# Rain risk scoring
Applying an expert rule-based system. It calculates a weighted risk score (0-100) and triggers a rain prediction if the score is $\ge$ 60.

In [ ]:
import numpy as np

scored_pandas = features_pandas.copy()

def normalize_0_100(series):
    min_value = series.min()
    max_value = series.max()

    if max_value == min_value:
        return pd.Series([50] * len(series), index=series.index)

    return 100 * (series - min_value) / (max_value - min_value)


scored_pandas["humidity_score"] = scored_pandas["humidity_roll_mean_3"].clip(0, 100)
scored_pandas["cloud_score"] = scored_pandas["cloud_cover_roll_mean_3"].clip(0, 100)

scored_pandas["pressure_drop"] = (-scored_pandas["pressure_change"]).clip(lower=0)
scored_pandas["pressure_drop_score"] = normalize_0_100(scored_pandas["pressure_drop"])

scored_pandas["previous_rain_score"] = np.where(
    scored_pandas["rain_previous"] > 0,
    100,
    0
)

scored_pandas["wind_score"] = normalize_0_100(scored_pandas["wind_speed_roll_mean_3"])

scored_pandas["rain_risk_score"] = (
    0.30 * scored_pandas["humidity_score"]
    + 0.25 * scored_pandas["cloud_score"]
    + 0.20 * scored_pandas["pressure_drop_score"]
    + 0.15 * scored_pandas["previous_rain_score"]
    + 0.10 * scored_pandas["wind_score"]
).round(2)

scored_pandas["risk_class"] = pd.cut(
    scored_pandas["rain_risk_score"],
    bins=[-1, 33, 66, 100],
    labels=["low", "medium", "high"]
)

scored_pandas["predicted_rain_rule"] = (
    scored_pandas["rain_risk_score"] >= 60
).astype(int)

print("Scoring finished.")
print("Rows:", len(scored_pandas))

scored_pandas[
    [
        "timestamp",
        "station_id",
        "rain_risk_score",
        "risk_class",
        "will_rain_next",
        "predicted_rain_rule"
    ]
].head(20)

In [ ]:
scored_key = "project12/output/rain_risk_scored.csv"

s3.put_object(
    Bucket=bucket,
    Key=scored_key,
    Body=scored_pandas.to_csv(index=False),
    ContentType="text/csv"
)

scored_path = f"s3://{bucket}/{scored_key}"

print("Scored data saved to:", scored_path)
print("Rows:", len(scored_pandas))

# Model evaluation
Calculating the Confusion Matrix components and standard classification metrics (Accuracy, Precision, Recall, F1-score).

In [ ]:
tp = len(scored_pandas[
    (scored_pandas["will_rain_next"] == 1) &
    (scored_pandas["predicted_rain_rule"] == 1)
])

tn = len(scored_pandas[
    (scored_pandas["will_rain_next"] == 0) &
    (scored_pandas["predicted_rain_rule"] == 0)
])

fp = len(scored_pandas[
    (scored_pandas["will_rain_next"] == 0) &
    (scored_pandas["predicted_rain_rule"] == 1)
])

fn = len(scored_pandas[
    (scored_pandas["will_rain_next"] == 1) &
    (scored_pandas["predicted_rain_rule"] == 0)
])

total = tp + tn + fp + fn

accuracy = (tp + tn) / total if total > 0 else 0
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1_score = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print("TP:", tp)
print("TN:", tn)
print("FP:", fp)
print("FN:", fn)

print("Accuracy:", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall:", round(recall, 4))
print("F1-score:", round(f1_score, 4))

## Metrics

In [ ]:
metrics_pandas = pd.DataFrame([
    ["TP", tp],
    ["TN", tn],
    ["FP", fp],
    ["FN", fn],
    ["Accuracy", round(accuracy, 4)],
    ["Precision", round(precision, 4)],
    ["Recall", round(recall, 4)],
    ["F1-score", round(f1_score, 4)]
], columns=["metric", "value"])

metrics_key = "project12/output/evaluation_metrics.csv"

s3.put_object(
    Bucket=bucket,
    Key=metrics_key,
    Body=metrics_pandas.to_csv(index=False),
    ContentType="text/csv"
)

metrics_path = f"s3://{bucket}/{metrics_key}"

print("Metrics saved to:", metrics_path)
metrics_pandas

## Risk summary

In [ ]:
risk_summary = scored_pandas.groupby("risk_class").agg(
    records_count=("rain_risk_score", "count"),
    avg_rain_risk_score=("rain_risk_score", "mean"),
    min_rain_risk_score=("rain_risk_score", "min"),
    max_rain_risk_score=("rain_risk_score", "max")
).reset_index()

risk_summary["avg_rain_risk_score"] = risk_summary["avg_rain_risk_score"].round(2)
risk_summary["min_rain_risk_score"] = risk_summary["min_rain_risk_score"].round(2)
risk_summary["max_rain_risk_score"] = risk_summary["max_rain_risk_score"].round(2)

risk_summary

In [ ]:
risk_summary_key = "project12/output/risk_summary.csv"

s3.put_object(
    Bucket=bucket,
    Key=risk_summary_key,
    Body=risk_summary.to_csv(index=False),
    ContentType="text/csv"
)

risk_summary_path = f"s3://{bucket}/{risk_summary_key}"

print("Risk summary saved to:", risk_summary_path)

In [ ]:
scored_excel_key = "project12/output/rain_risk_scored_excel.csv"

scored_pandas.to_csv(
    "/tmp/rain_risk_scored_excel.csv",
    index=False,
    sep=";",
    encoding="utf-8-sig"
)

with open("/tmp/rain_risk_scored_excel.csv", "rb") as f:
    s3.put_object(
        Bucket=bucket,
        Key=scored_excel_key,
        Body=f.read(),
        ContentType="text/csv"
    )

scored_excel_path = f"s3://{bucket}/{scored_excel_key}"

print("Excel CSV saved to:", scored_excel_path)
print("Rows:", len(scored_pandas))

## Prediction summary

In [ ]:
prediction_summary = scored_pandas.groupby(
    ["predicted_rain_rule", "will_rain_next"]
).size().reset_index(name="records_count")

prediction_summary

In [ ]:
prediction_summary_key = "project12/output/prediction_summary.csv"

s3.put_object(
    Bucket=bucket,
    Key=prediction_summary_key,
    Body=prediction_summary.to_csv(index=False),
    ContentType="text/csv"
)

prediction_summary_path = f"s3://{bucket}/{prediction_summary_key}"

print("Prediction summary saved to:", prediction_summary_path)